# NarrativeService Interactive Testing

This notebook allows you to interactively test the `NarrativeService`. It sets up the Django environment, creates dummy data, and then runs the service to hit the LLM via `LLMProvider`.

In [ ]:
import os
import django

# Setup Django environment
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'config.settings')
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

# TODO: Set your Groq API key here before running the LLM cell
os.environ["GROQ_API_KEY"] = "***"

django.setup()
print("Django environment initialized.")

Django environment initialized.


### Create Dummy Data
We need `ReconciliationReport` and `AnalyticsReport` objects in the database.

In [2]:
from datetime import date
from core.models.reconciliation import ReconciliationReport
from core.models.analytics import AnalyticsReport, HourlyRevenue, MedicineRankEntry

# Clean up previous dummy data to avoid UniqueConstraint errors across runs
ReconciliationReport.objects.filter(clinic_id="TEST-CLN-001").delete()
AnalyticsReport.objects.filter(clinic_id="TEST-CLN-001").delete()

# 1. Create a dummy ReconciliationReport
recon = ReconciliationReport.objects.create(
    clinic_id="TEST-CLN-001",
    report_date=date.today(),
    total_billed_paise=4285000,
    total_collected_paise=4000000,
    total_outstanding_paise=285000,
    total_refunds_paise=0,
    visit_count=10,
    refund_count=0
)

# 2. Create a dummy AnalyticsReport with related data
analytics = AnalyticsReport.objects.create(
    clinic_id="TEST-CLN-001",
    report_date=date.today()
)

HourlyRevenue.objects.create(analytics_report=analytics, hour=12, revenue_paise=1500000)
MedicineRankEntry.objects.create(
    analytics_report=analytics,
    rank_type=MedicineRankEntry.RANK_TYPE_QUANTITY,
    drug_name="PARACETAMOL",
    value=20,
    rank=1
)
MedicineRankEntry.objects.create(
    analytics_report=analytics,
    rank_type=MedicineRankEntry.RANK_TYPE_REVENUE,
    drug_name="AMOXICILLIN",
    value=800000,
    rank=1
)
print("Dummy data created successfully!")

Dummy data created successfully!


### Test Narrative Context Builder

In [3]:
from core.llm_provider import LLMProvider
from core.narrative_service import NarrativeService

# Initialize the LLM Provider
api_key = os.environ.get("GROQ_API_KEY", "")
llm = LLMProvider(api_key=api_key)

# Initialize the Narrative Service
narrative_service = NarrativeService(llm_provider=llm)

# Test context building
context = narrative_service.build_context(recon, analytics)
print("--- Generated Context ---")
for key, val in context.values.items():
    print(f"{{{{{key}}}}}: {val}")

--- Generated Context ---
{{total_billed}}: ₹42,850
{{total_collected}}: ₹40,000
{{total_outstanding}}: ₹2,850
{{total_refunds}}: ₹0
{{visit_count}}: 10
{{refund_count}}: 0
{{peak_hour_revenue}}: ₹15,000
{{peak_hour_time}}: 12pm–1pm
{{top_medicine_by_qty_name}}: PARACETAMOL
{{top_medicine_by_qty_value}}: 20
{{top_medicine_by_rev_name}}: AMOXICILLIN
{{top_medicine_by_rev_value}}: ₹8,000


### Call LLM & Grounding Validations
**Note:** Ensure your `GROQ_API_KEY` is correctly set in the first cell, otherwise you will receive a 401 error.

In [4]:
print("Calling LLM API to generate narrative...")

# Test full generation pipeline
result = narrative_service.generate_narrative(recon, analytics)

print(f"\nStatus: {result.status}")
print("\n--- Final Narrative Text ---")
print(result.text)

print("\n--- Traced Figures ---")
for figure in getattr(result, "_unsaved_figures", []):
    print(f"{figure.placeholder} -> {figure.display_value}")
    
if result.warnings:
    print("\n--- Warnings ---")
    for warning in result.warnings:
        print(warning)

Calling LLM API to generate narrative...

Status: SUCCESS

--- Final Narrative Text ---
Billing Report Summary for 10 visits:

   Total Billed: ₹42,850

   Total Collected: ₹40,000

   Total Outstanding: ₹2,850

   Total Refunds: ₹0

   Number of Refunds: 0

   Peak Hour Revenue: ₹15,000 at 12pm–1pm

   Top Medicine by Quantity: PARACETAMOL with 20 units sold

   Top Medicine by Revenue: AMOXICILLIN with ₹8,000 revenue generated
Note: cost data wasn't available today, so this is revenue, not profit.

--- Traced Figures ---
peak_hour_revenue -> ₹15,000
top_medicine_by_rev_name -> AMOXICILLIN
top_medicine_by_rev_value -> ₹8,000
top_medicine_by_qty_value -> 20
peak_hour_time -> 12pm–1pm
refund_count -> 0
total_billed -> ₹42,850
total_collected -> ₹40,000
visit_count -> 10
top_medicine_by_qty_name -> PARACETAMOL
total_refunds -> ₹0
total_outstanding -> ₹2,850
